# Thermal-IR Drone and Bird Detection Pipeline

**AAI-590 Capstone | Stage 2 — dataset preparation**

This notebook is the reproducible entry point for the thermal-IR workflow. It prepares and validates data only; it does not train a model unless the explicit training gate near the end is approved.

- Class `0`: Drone
- Class `1`: Bird
- Source images and videos remain immutable.
- Splits must be recording/sequence-disjoint.
- Dataset 1 supplied test folders are inference-only and are excluded from metrics.

## Dataset sources and current audit status

| Source | Intended use | Local raw destination | Status |
| --- | --- | --- | --- |
| Official 4th Anti-UAV Challenge / Zenodo 15103888 | Additional thermal drone examples | `data/thermal-ir/raw/anti_uav_dataset1/` | Dataset 1 `train` is frame-labelled; supplied test folders are inference-only |
| DroneDetectionThesis/Drone-detection-dataset | Thermal drone and bird examples | `data/thermal-ir/raw/drone_detection_dataset2/` | `.mat` labels require `mcos-decoder`; video/label alignment needs validation |

Read the complete Stage 1 findings before processing: [`docs/thermal_ir_stage1_dataset_audit.md`](../docs/thermal_ir_stage1_dataset_audit.md).

Dataset 2 is present locally. Dataset 1 is being copied manually; do not execute Dataset 1 manifest, conversion, or split cells until its raw-file gate reports complete.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

# Support running from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY
elif (WORKING_DIRECTORY.parent / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY.parent
else:
    raise RuntimeError('Open this notebook from the AAI-590 repository or its notebooks/ directory.')

THERMAL_ROOT = REPO_ROOT / 'data' / 'thermal-ir'
RAW_ROOT = THERMAL_ROOT / 'raw'
DATASET1_ROOT = RAW_ROOT / 'anti_uav_dataset1'
DATASET2_ROOT = RAW_ROOT / 'drone_detection_dataset2' / 'Data' / 'Video_IR'
MANIFEST_ROOT = THERMAL_ROOT / 'manifests'
QA_ROOT = THERMAL_ROOT / 'interim' / 'qa_overlays'
YOLO_ROOT = THERMAL_ROOT / 'processed' / 'yolo_v1'
RESULTS_ROOT = REPO_ROOT / 'results' / 'thermal-ir'

CLASS_NAMES = {0: 'Drone', 1: 'Bird'}
RANDOM_SEED = 590

print(f'Python: {sys.version.split()[0]}')
print(f'Repository: {REPO_ROOT}')
print(f'Thermal data root: {THERMAL_ROOT}')

Python: 3.11.7
Repository: D:\Capstone\aai-590-capstone
Thermal data root: D:\Capstone\aai-590-capstone\data\thermal-ir


## Environment setup

Create a dedicated environment from the repository root before running processing cells:

```powershell
uv venv .venv
.\.venv\Scripts\Activate.ps1
uv pip install pandas pillow opencv-python-headless mcos-decoder pyyaml matplotlib seaborn
```

`ultralytics` is intentionally omitted at this stage. Add it only after the prepared dataset and training configuration are approved.

In [2]:
from importlib.util import find_spec

REQUIRED_PACKAGES = {
    'pandas': 'pandas',
    'Pillow': 'PIL',
    'OpenCV': 'cv2',
    'mcos-decoder': 'mcos_decoder',
    'PyYAML': 'yaml',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
}
missing_packages = [name for name, module in REQUIRED_PACKAGES.items() if find_spec(module) is None]
if missing_packages:
    print('Missing packages:', ', '.join(missing_packages))
    print('Install them using the previous cell before running data-processing cells.')
else:
    print('All Stage 2 processing dependencies are available.')

All Stage 2 processing dependencies are available.


In [3]:
# These expected counts were established in the Stage 1 read-only audit.
EXPECTED_RAW_FILE_COUNTS = {
    'Dataset 1 Anti-UAV': 763_717,
    'Dataset 2 Drone Detection': 736,
}
RAW_DATASET_ROOTS = {
    'Dataset 1 Anti-UAV': DATASET1_ROOT,
    'Dataset 2 Drone Detection': RAW_ROOT / 'drone_detection_dataset2',
}

def count_files(path: Path) -> int:
    """Count dataset payload files, excluding copied Git metadata."""
    return sum(
        1
        for candidate in path.rglob('*')
        if candidate.is_file() and '.git' not in candidate.relative_to(path).parts
    )

def raw_copy_status() -> dict[str, dict[str, object]]:
    report = {}
    for name, path in RAW_DATASET_ROOTS.items():
        actual = count_files(path) if path.is_dir() else 0
        expected = EXPECTED_RAW_FILE_COUNTS[name]
        report[name] = {
            'path': path,
            'actual_files': actual,
            'expected_files': expected,
            'complete': actual == expected,
        }
    return report

RAW_STATUS = raw_copy_status()
for name, status in RAW_STATUS.items():
    label = 'COMPLETE' if status['complete'] else 'INCOMPLETE'
    print(f"{label:>10}  {name}: {status['actual_files']:,}/{status['expected_files']:,} payload files")

def require_complete_raw_data() -> None:
    incomplete = [name for name, status in RAW_STATUS.items() if not status['complete']]
    if incomplete:
        raise RuntimeError(
            'Raw copy is incomplete for: ' + ', '.join(incomplete) +
            '. Complete the manual copy, re-run the raw-completeness cell, then continue.'
        )

  COMPLETE  Dataset 1 Anti-UAV: 763,717/763,717 payload files
  COMPLETE  Dataset 2 Drone Detection: 736/736 payload files


In [4]:
# Read-only preflight: this cell deliberately does not create, copy, or modify data.
for name, path in {
    'Dataset 1 Anti-UAV train': DATASET1_ROOT / 'train',
    'Dataset 2 Video_IR': DATASET2_ROOT,
}.items():
    state = 'READY' if path.is_dir() else 'NOT YET POPULATED'
    print(f'{state:>18}  {name}: {path}')

print('\nThe raw-completeness gate must pass before manifest generation or extraction.')

             READY  Dataset 1 Anti-UAV train: D:\Capstone\aai-590-capstone\data\thermal-ir\raw\anti_uav_dataset1\train
             READY  Dataset 2 Video_IR: D:\Capstone\aai-590-capstone\data\thermal-ir\raw\drone_detection_dataset2\Data\Video_IR

The raw-completeness gate must pass before manifest generation or extraction.


## Stage 2A — read-only source manifests and annotation validation

Run the following cells only after the raw-copy gate is complete. They read immutable source files and write only derived CSV manifests under `data/thermal-ir/manifests/`. No source image, video, JSON, or MATLAB file is modified.

The issue log is intentionally comprehensive: invalid boxes, missing paired files, array-length mismatches, and Dataset 2 annotation/video frame-count mismatches are all retained rather than silently corrected.

In [5]:
import json
from collections import Counter

import cv2
import pandas as pd
from mcos_decoder import load_groundtruth
from PIL import Image

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
DATASET2_EXCLUDED_CLASSES = {'AIRPLANE', 'HELICOPTER'}
DATASET2_ELIGIBLE_CLASS_IDS = {'DRONE': 0, 'BIRD': 1}

def source_frames(sequence_dir: Path) -> list[Path]:
    """Return deterministic, frame-number ordered source images for one Dataset 1 sequence."""
    return sorted(
        (path for path in sequence_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES),
        key=lambda path: path.name,
    )

def xywh_issue(box, width: int, height: int) -> str | None:
    """Validate a pixel-space top-left xywh box without changing it."""
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return 'invalid_box_shape'
    try:
        x, y, box_width, box_height = (float(value) for value in box)
    except (TypeError, ValueError):
        return 'non_numeric_box'
    if not all(pd.notna(value) for value in (x, y, box_width, box_height)):
        return 'non_finite_box'
    if box_width <= 0 or box_height <= 0:
        return 'non_positive_box_dimensions'
    if x < 0 or y < 0:
        return 'negative_box_origin'
    if x + box_width > width or y + box_height > height:
        return 'box_exceeds_image_boundary'
    return None

def manifest_issue(source_dataset: str, source_sequence: str, issue_type: str, **details) -> dict:
    return {
        'source_dataset': source_dataset,
        'source_sequence': source_sequence,
        'issue_type': issue_type,
        **details,
    }

In [6]:
def inspect_dataset1_train() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Inspect Dataset 1 `train` only; do not alter its frames or annotations."""
    sequence_rows, issues = [], []
    train_root = DATASET1_ROOT / 'train'
    for sequence_dir in sorted(path for path in train_root.iterdir() if path.is_dir()):
        sequence_id = sequence_dir.name
        labels_path = sequence_dir / 'IR_label.json'
        frames = source_frames(sequence_dir)
        row = {
            'source_dataset': 'anti_uav_dataset1',
            'source_partition': 'train',
            'source_sequence': sequence_id,
            'frame_count': len(frames),
            'annotation_path': labels_path.relative_to(REPO_ROOT).as_posix(),
        }
        if not frames:
            issues.append(manifest_issue('anti_uav_dataset1', sequence_id, 'missing_source_frames'))
            sequence_rows.append(row)
            continue
        with Image.open(frames[0]) as first_frame:
            width, height = first_frame.size
        row.update({'image_width': width, 'image_height': height})
        if not labels_path.is_file():
            issues.append(manifest_issue('anti_uav_dataset1', sequence_id, 'missing_annotation_file'))
            sequence_rows.append(row)
            continue
        try:
            labels = json.loads(labels_path.read_text(encoding='utf-8'))
            exists, boxes = labels.get('exist'), labels.get('gt_rect')
            if not isinstance(exists, list) or not isinstance(boxes, list):
                raise ValueError('missing `exist` or `gt_rect` list')
        except (OSError, ValueError, json.JSONDecodeError) as error:
            issues.append(manifest_issue('anti_uav_dataset1', sequence_id, 'unreadable_annotation', detail=str(error)))
            sequence_rows.append(row)
            continue
        row.update({
            'exist_entries': len(exists),
            'box_entries': len(boxes),
            'visible_entries': sum(value == 1 for value in exists),
            'absent_entries': sum(value != 1 for value in exists),
            'annotation_keys': ','.join(sorted(labels)),
        })
        if len(frames) != len(exists) or len(exists) != len(boxes):
            issues.append(manifest_issue(
                'anti_uav_dataset1', sequence_id, 'frame_annotation_length_mismatch',
                frame_count=len(frames), exist_entries=len(exists), box_entries=len(boxes),
            ))
        for frame_index, (exists_flag, box) in enumerate(zip(exists, boxes), start=1):
            if exists_flag == 1:
                issue_type = xywh_issue(box, width, height)
                if issue_type:
                    issues.append(manifest_issue(
                        'anti_uav_dataset1', sequence_id, issue_type,
                        original_frame=frame_index, bbox=json.dumps(box),
                    ))
        sequence_rows.append(row)
    return pd.DataFrame(sequence_rows), pd.DataFrame(issues)

In [7]:
def inspect_dataset2_videos() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Inspect all Dataset 2 IR pairs and retain excluded-class inventory information."""
    video_rows, issues = [], []
    for labels_path in sorted(DATASET2_ROOT.glob('IR_*_LABELS.mat')):
        parts = labels_path.stem.split('_')
        source_class = parts[1] if len(parts) >= 3 else 'UNKNOWN'
        recording_id = labels_path.name.removesuffix('_LABELS.mat')
        video_path = labels_path.with_name(f'{recording_id}.mp4')
        row = {
            'source_dataset': 'drone_detection_dataset2',
            'source_sequence': recording_id,
            'source_class': source_class,
            'eligible_for_two_class_pipeline': source_class in DATASET2_ELIGIBLE_CLASS_IDS,
            'class_id': DATASET2_ELIGIBLE_CLASS_IDS.get(source_class),
            'video_path': video_path.relative_to(REPO_ROOT).as_posix(),
            'annotation_path': labels_path.relative_to(REPO_ROOT).as_posix(),
        }
        if not video_path.is_file():
            issues.append(manifest_issue('drone_detection_dataset2', recording_id, 'missing_video_file'))
            video_rows.append(row)
            continue
        capture = cv2.VideoCapture(str(video_path))
        try:
            frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = float(capture.get(cv2.CAP_PROP_FPS))
        finally:
            capture.release()
        boxes = load_groundtruth(labels_path)
        visible_boxes = [box for box in boxes if box is not None]
        row.update({
            'video_frame_count': frame_count,
            'annotation_entries': len(boxes),
            'visible_entries': len(visible_boxes),
            'absent_entries': len(boxes) - len(visible_boxes),
            'image_width': width,
            'image_height': height,
            'fps': fps,
        })
        if frame_count != len(boxes):
            issues.append(manifest_issue(
                'drone_detection_dataset2', recording_id, 'video_annotation_length_mismatch',
                video_frame_count=frame_count, annotation_entries=len(boxes),
            ))
        for frame_index, box in enumerate(boxes, start=1):
            if box is not None:
                issue_type = xywh_issue(box, width, height)
                if issue_type:
                    issues.append(manifest_issue(
                        'drone_detection_dataset2', recording_id, issue_type,
                        original_frame=frame_index, bbox=json.dumps(box),
                    ))
        video_rows.append(row)
    return pd.DataFrame(video_rows), pd.DataFrame(issues)

In [8]:
# Execute this cell after the raw-copy gate reports COMPLETE for both sources.
# It creates only derived CSVs; raw sources remain untouched.
require_complete_raw_data()
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

dataset1_manifest, dataset1_issues = inspect_dataset1_train()
dataset2_manifest, dataset2_issues = inspect_dataset2_videos()
issues = pd.concat([dataset1_issues, dataset2_issues], ignore_index=True)

dataset1_manifest.to_csv(MANIFEST_ROOT / 'dataset1_sequences.csv', index=False)
dataset2_manifest.to_csv(MANIFEST_ROOT / 'dataset2_videos.csv', index=False)
issues.to_csv(MANIFEST_ROOT / 'annotation_issues.csv', index=False)

print(f'Dataset 1 sequences: {len(dataset1_manifest):,}')
print(f'Dataset 2 videos: {len(dataset2_manifest):,}')
print('Issue counts:')
display(issues['issue_type'].value_counts(dropna=False).rename_axis('issue_type').to_frame('count'))

Dataset 1 sequences: 223
Dataset 2 videos: 365
Issue counts:


,count
issue_type,
box_exceeds_image_boundary,130
non_positive_box_dimensions,121
video_annotation_length_mismatch,85


## Stage 2B — deterministic visual QA overlays

Review representative valid targets and every discovered issue category before exporting any YOLO labels. These images are inspection artifacts only; the source frames remain unchanged and no overlay is used for model training.

In [9]:
from PIL import ImageDraw, ImageFont

def parse_bbox(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, str):
        value = json.loads(value)
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        return None
    return tuple(float(component) for component in value)

def render_overlay(image, bbox, caption, output_path):
    rendered = image.convert('RGB').copy()
    draw = ImageDraw.Draw(rendered)
    if bbox is not None:
        x, y, width, height = bbox
        draw.rectangle((x, y, x + width, y + height), outline='lime', width=3)
    font = ImageFont.load_default()
    bounds = draw.textbbox((0, 0), caption, font=font)
    draw.rectangle((4, 4, bounds[2] + 12, bounds[3] + 12), fill='black')
    draw.text((8, 8), caption, fill='lime', font=font)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rendered.save(output_path)

def read_dataset2_frame(recording_id, frame_number):
    capture = cv2.VideoCapture(str(DATASET2_ROOT / f'{recording_id}.mp4'))
    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number - 1)
        ok, frame = capture.read()
    finally:
        capture.release()
    if not ok:
        raise RuntimeError(f'Unable to decode {recording_id} frame {frame_number}')
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

In [10]:
# Execute after `write-manifests`. This creates derived QA images and an index only.
dataset1_manifest = pd.read_csv(MANIFEST_ROOT / 'dataset1_sequences.csv')
dataset2_manifest = pd.read_csv(MANIFEST_ROOT / 'dataset2_videos.csv')
issues = pd.read_csv(MANIFEST_ROOT / 'annotation_issues.csv')
qa_records = []

def add_overlay(source_dataset, sequence, frame_number, class_name, bbox, sample_kind, issue_type=None):
    number = len(qa_records) + 1
    filename = f'{number:02d}_{source_dataset}_{sequence}_{frame_number:06d}.png'
    output_path = QA_ROOT / filename
    caption = f'{class_name} | {sample_kind} | frame {frame_number}'
    if issue_type:
        caption += f' | {issue_type}'
    if source_dataset == 'anti_uav_dataset1':
        frame_path = source_frames(DATASET1_ROOT / 'train' / sequence)[frame_number - 1]
        with Image.open(frame_path) as image:
            render_overlay(image, bbox, caption, output_path)
    else:
        render_overlay(read_dataset2_frame(sequence, frame_number), bbox, caption, output_path)
    qa_records.append({
        'source_dataset': source_dataset, 'source_sequence': sequence,
        'original_frame': frame_number, 'class_name': class_name,
        'sample_kind': sample_kind, 'issue_type': issue_type,
        'bbox': json.dumps(bbox) if bbox is not None else None,
        'output_path': output_path.relative_to(REPO_ROOT).as_posix(),
    })

# Three valid Dataset 1 drone examples.
for _, row in dataset1_manifest.sort_values('source_sequence').iloc[:3].iterrows():
    labels = json.loads((REPO_ROOT / row.annotation_path).read_text(encoding='utf-8'))
    for frame_number, (exists_flag, bbox) in enumerate(zip(labels['exist'], labels['gt_rect']), start=1):
        if exists_flag == 1 and xywh_issue(bbox, row.image_width, row.image_height) is None:
            add_overlay('anti_uav_dataset1', row.source_sequence, frame_number, 'Drone', parse_bbox(bbox), 'valid_target')
            break

# Three valid Dataset 2 examples for each eligible class.
for source_class, class_name in (('DRONE', 'Drone'), ('BIRD', 'Bird')):
    for _, row in dataset2_manifest.query('source_class == @source_class').sort_values('source_sequence').head(3).iterrows():
        boxes = load_groundtruth(REPO_ROOT / row.annotation_path)
        for frame_number, bbox in enumerate(boxes, start=1):
            if bbox is not None and xywh_issue(bbox, row.image_width, row.image_height) is None:
                add_overlay('drone_detection_dataset2', row.source_sequence, frame_number, class_name, parse_bbox(bbox), 'valid_target')
                break

# Two examples for each Dataset 1 invalid-box category.
for issue_type in ('box_exceeds_image_boundary', 'non_positive_box_dimensions'):
    for _, row in issues.query("source_dataset == 'anti_uav_dataset1' and issue_type == @issue_type").head(2).iterrows():
        add_overlay('anti_uav_dataset1', row.source_sequence, int(row.original_frame), 'Drone', parse_bbox(row.bbox), 'annotation_issue', issue_type)

# Dataset 2 mismatches: render the last frame in the validated video/label overlap.
for _, issue in issues.query("source_dataset == 'drone_detection_dataset2' and issue_type == 'video_annotation_length_mismatch'").head(3).iterrows():
    row = dataset2_manifest.loc[dataset2_manifest.source_sequence == issue.source_sequence].iloc[0]
    frame_number = min(int(row.video_frame_count), int(row.annotation_entries))
    boxes = load_groundtruth(REPO_ROOT / row.annotation_path)
    bbox = boxes[frame_number - 1] if frame_number <= len(boxes) else None
    class_name = 'Drone' if row.source_class == 'DRONE' else 'Bird'
    add_overlay('drone_detection_dataset2', row.source_sequence, frame_number, class_name, parse_bbox(bbox), 'overlap_boundary', 'video_annotation_length_mismatch')

qa_index = pd.DataFrame(qa_records)
qa_index.to_csv(MANIFEST_ROOT / 'qa_sample_index.csv', index=False)
print(f'Rendered {len(qa_index)} QA overlays to {QA_ROOT}')
display(qa_index)

Rendered 16 QA overlays to D:\Capstone\aai-590-capstone\data\thermal-ir\interim\qa_overlays


,source_dataset,source_sequence,original_frame,class_name,sample_kind,issue_type,bbox,output_path
0,anti_uav_dataset1,01_1751_0250-1750,1,Drone,valid_target,NaN,"[306.0, 270.0, 14.0, 11.0]",data/thermal-ir/interim/qa_overlays/01_anti_ua...
1,anti_uav_dataset1,01_2192_0001-1500,1,Drone,valid_target,NaN,"[268.0, 279.0, 29.0, 22.0]",data/thermal-ir/interim/qa_overlays/02_anti_ua...
2,anti_uav_dataset1,01_4485_1167-2666,4,Drone,valid_target,NaN,"[605.0, 361.0, 27.0, 10.0]",data/thermal-ir/interim/qa_overlays/03_anti_ua...
3,drone_detection_dataset2,IR_DRONE_001,1,Drone,valid_target,NaN,"[26.76181411743164, 118.27247619628906, 28.563...",data/thermal-ir/interim/qa_overlays/04_drone_d...
4,drone_detection_dataset2,IR_DRONE_002,1,Drone,valid_target,NaN,"[165.0, 113.0, 29.0, 13.0]",data/thermal-ir/interim/qa_overlays/05_drone_d...
5,drone_detection_dataset2,IR_DRONE_003,1,Drone,valid_target,NaN,"[177.0, 144.0, 24.0, 13.0]",data/thermal-ir/interim/qa_overlays/06_drone_d...
6,drone_detection_dataset2,IR_BIRD_001,2,Bird,valid_target,NaN,"[3.0, 153.0, 8.0, 9.0]",data/thermal-ir/interim/qa_overlays/07_drone_d...
7,drone_detection_dataset2,IR_BIRD_002,2,Bird,valid_target,NaN,"[120.0, 136.0, 9.0, 9.0]",data/thermal-ir/interim/qa_overlays/08_drone_d...
8,drone_detection_dataset2,IR_BIRD_003,2,Bird,valid_target,NaN,"[158.0, 141.0, 13.0, 14.0]",data/thermal-ir/interim/qa_overlays/09_drone_d...
9,anti_uav_dataset1,01_4485_1167-2666,1,Drone,annotation_issue,box_exceeds_image_boundary,"[633.0, 365.0, 8.0, 8.0]",data/thermal-ir/interim/qa_overlays/10_anti_ua...


## Stage 5 — PyTorch YOLO model development (approval-gated)

The detector will use a lightweight pretrained Ultralytics YOLO detection model, which runs on PyTorch and CUDA without Docker. Use **one GPU** (`device=0`) on Windows. The model section remains blocked until the conversion, deduplication, and recording-disjoint split stages produce a validated `data.yaml`, frame metadata, and split-integrity report.

The baseline model is `yolo26n.pt`: the nano variant limits initial compute cost while pretrained weights provide a practical transfer-learning baseline. Its suitability must be reported alongside limitations for small thermal targets; it is not an operational-system validation.

Install the CUDA-enabled PyTorch build selected for the local driver from [PyTorch's official selector](https://pytorch.org/get-started/locally/), then install Ultralytics:

```powershell
# Run from the repository root after activating .venv.
# Use the exact CUDA wheel command supplied by pytorch.org for this system.
uv pip install torch torchvision --index-url <official-pytorch-cuda-wheel-index>
uv pip install ultralytics
```

No Docker configuration is required.

In [11]:
# Execute immediately before training; it fails fast if CUDA PyTorch is unavailable.
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Install the CUDA-enabled PyTorch wheel selected from pytorch.org, '
        'then restart the notebook kernel.'
    )
print(f'PyTorch: {torch.__version__}')
print(f'CUDA runtime: {torch.version.cuda}')
print(f'Device 0: {torch.cuda.get_device_name(0)}')
print(f'Available memory (GiB): {torch.cuda.mem_get_info(0)[0] / 2**30:.1f}')

PyTorch: 2.14.1+cu130
CUDA runtime: 13.0
Device 0: NVIDIA GeForce RTX 5080
Available memory (GiB): 14.6


In [12]:
# Fixed initial configuration for a controlled Experiment A/B comparison.
# Changes after this point require a documented experiment revision.
MODEL_CONFIG = {
    'model': 'yolo26n.pt',
    'imgsz': 640,
    'epochs': 100,
    'batch': 16,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 1e-3,
    'weight_decay': 5e-4,
    'patience': 20,
    'seed': RANDOM_SEED,
    'deterministic': True,
    'workers': 8,
    'pretrained': True,
    'cache': False,
    # Thermal-appropriate geometric augmentation; disable hue/saturation/value color changes.
    'hsv_h': 0.0,
    'hsv_s': 0.0,
    'hsv_v': 0.0,
    'fliplr': 0.5,
    'flipud': 0.0,
    'degrees': 0.0,
    'translate': 0.10,
    'scale': 0.25,
    'mosaic': 0.5,
    'close_mosaic': 10,
}

EXPERIMENTS = {
    'experiment_a_dataset2_only': YOLO_ROOT / 'experiment_a_dataset2_only.yaml',
    'experiment_b_combined': YOLO_ROOT / 'experiment_b_combined.yaml',
}
TRAINING_RUN_ROOT = RESULTS_ROOT / 'runs'
TEST_IOU_THRESHOLD = 0.50
TEST_CONFIDENCE_THRESHOLD = 0.001

assert CLASS_NAMES == {0: 'Drone', 1: 'Bird'}
MODEL_CONFIG

{'model': 'yolo26n.pt',
 'imgsz': 640,
 'epochs': 100,
 'batch': 16,
 'device': 0,
 'optimizer': 'AdamW',
 'lr0': 0.001,
 'weight_decay': 0.0005,
 'patience': 20,
 'seed': 590,
 'deterministic': True,
 'workers': 8,
 'pretrained': True,
 'cache': False,
 'hsv_h': 0.0,
 'hsv_s': 0.0,
 'hsv_v': 0.0,
 'fliplr': 0.5,
 'flipud': 0.0,
 'degrees': 0.0,
 'translate': 0.1,
 'scale': 0.25,
 'mosaic': 0.5,
 'close_mosaic': 10}

In [13]:
def require_prepared_yolo_dataset(experiment_yaml: Path) -> None:
    required = [
        experiment_yaml,
        YOLO_ROOT / 'metadata' / 'frames.csv',
        YOLO_ROOT / 'metadata' / 'split_integrity_report.json',
    ]
    missing = [str(path.relative_to(REPO_ROOT)) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(
            'Training is blocked until the validated exported dataset exists. Missing: ' + ', '.join(missing)
        )

def train_experiment(experiment_name: str):
    """Train one controlled experiment; never use the held-out test split for model selection."""
    if experiment_name not in EXPERIMENTS:
        raise KeyError(f'Unknown experiment: {experiment_name}')
    data_yaml = EXPERIMENTS[experiment_name]
    require_prepared_yolo_dataset(data_yaml)
    run_directory = TRAINING_RUN_ROOT / experiment_name
    model = YOLO(MODEL_CONFIG['model'])
    return model.train(
        data=str(data_yaml),
        project=str(run_directory.parent),
        name=run_directory.name,
        exist_ok=False,
        **{key: value for key, value in MODEL_CONFIG.items() if key != 'model'},
    )

# After approval and dataset preparation, run exactly one at a time:
# train_experiment('experiment_a_dataset2_only')
# train_experiment('experiment_b_combined')

## Held-out test evaluation

After validation-based model selection is finished, evaluate the selected checkpoint once on the isolated test split. Ultralytics validation supplies precision, recall, mAP@0.5, mAP@0.5:0.95, per-class AP, curves, and a detection confusion matrix. The custom exporter below supplements those artifacts with an auditable row-level test-predictions CSV.

A prediction is matched greedily to the highest-IoU unused ground-truth object at IoU ≥ 0.50. Rows are labeled `true_positive`, `class_mismatch`, `false_negative`, or `false_positive`; that distinction prevents confusing detection error analysis with image-level classification.

In [14]:
import numpy as np

def iou_xyxy(first: np.ndarray, second: np.ndarray) -> float:
    left, top = np.maximum(first[:2], second[:2])
    right, bottom = np.minimum(first[2:], second[2:])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    first_area = max(0.0, first[2] - first[0]) * max(0.0, first[3] - first[1])
    second_area = max(0.0, second[2] - second[0]) * max(0.0, second[3] - second[1])
    union = first_area + second_area - intersection
    return intersection / union if union else 0.0

def yolo_label_to_xyxy(parts: list[float], width: int, height: int) -> tuple[int, np.ndarray]:
    class_id, x_center, y_center, box_width, box_height = parts
    x_center, box_width = x_center * width, box_width * width
    y_center, box_height = y_center * height, box_height * height
    return int(class_id), np.array([x_center - box_width / 2, y_center - box_height / 2, x_center + box_width / 2, y_center + box_height / 2])

def read_ground_truth(label_path: Path, width: int, height: int):
    if not label_path.is_file() or not label_path.read_text().strip():
        return []
    return [yolo_label_to_xyxy([float(value) for value in line.split()], width, height) for line in label_path.read_text().splitlines()]

def export_heldout_predictions(best_weights: Path, experiment_yaml: Path) -> pd.DataFrame:
    """Run final inference once and write row-level matched/unmatched test outcomes."""
    require_prepared_yolo_dataset(experiment_yaml)
    metadata = pd.read_csv(YOLO_ROOT / 'metadata' / 'frames.csv')
    test_rows = metadata.query("split == 'test'").copy()
    if test_rows.empty:
        raise RuntimeError('No held-out test rows found in frames.csv.')
    model = YOLO(str(best_weights))
    results = model.val(data=str(experiment_yaml), split='test', plots=True, conf=TEST_CONFIDENCE_THRESHOLD)
    metrics_path = RESULTS_ROOT / 'metrics' / f'{best_weights.parent.parent.name}_test_metrics.json'
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(results.results_dict, indent=2), encoding='utf-8')
    exported = []
    for _, row in test_rows.iterrows():
        image_path = REPO_ROOT / row.image_path
        label_path = REPO_ROOT / row.label_path
        with Image.open(image_path) as image:
            width, height = image.size
        ground_truth = read_ground_truth(label_path, width, height)
        prediction = model.predict(str(image_path), conf=TEST_CONFIDENCE_THRESHOLD, verbose=False)[0]
        predicted = [
            (int(class_id), float(confidence), box.cpu().numpy())
            for class_id, confidence, box in zip(prediction.boxes.cls, prediction.boxes.conf, prediction.boxes.xyxy)
        ]
        pairs = sorted(
            ((iou_xyxy(gt_box, pred_box), gt_index, pred_index) for gt_index, (_, gt_box) in enumerate(ground_truth) for pred_index, (_, _, pred_box) in enumerate(predicted)),
            reverse=True,
        )
        matched_gt, matched_pred = set(), set()
        for iou, gt_index, pred_index in pairs:
            if iou < TEST_IOU_THRESHOLD or gt_index in matched_gt or pred_index in matched_pred:
                continue
            matched_gt.add(gt_index); matched_pred.add(pred_index)
            gt_class, gt_box = ground_truth[gt_index]
            pred_class, confidence, pred_box = predicted[pred_index]
            exported.append({**row.to_dict(), 'ground_truth_class_id': gt_class, 'predicted_class_id': pred_class, 'confidence': confidence, 'iou': iou, 'outcome': 'true_positive' if gt_class == pred_class else 'class_mismatch', 'ground_truth_xyxy': gt_box.tolist(), 'predicted_xyxy': pred_box.tolist()})
        for gt_index, (gt_class, gt_box) in enumerate(ground_truth):
            if gt_index not in matched_gt:
                exported.append({**row.to_dict(), 'ground_truth_class_id': gt_class, 'predicted_class_id': None, 'confidence': None, 'iou': 0.0, 'outcome': 'false_negative', 'ground_truth_xyxy': gt_box.tolist(), 'predicted_xyxy': None})
        for pred_index, (pred_class, confidence, pred_box) in enumerate(predicted):
            if pred_index not in matched_pred:
                exported.append({**row.to_dict(), 'ground_truth_class_id': None, 'predicted_class_id': pred_class, 'confidence': confidence, 'iou': 0.0, 'outcome': 'false_positive', 'ground_truth_xyxy': None, 'predicted_xyxy': pred_box.tolist()})
    report = pd.DataFrame(exported)
    output = RESULTS_ROOT / 'test_predictions.csv'
    output.parent.mkdir(parents=True, exist_ok=True)
    report.to_csv(output, index=False)
    return report

# After selecting a validation-approved checkpoint, run once:
# test_report = export_heldout_predictions(Path('.../weights/best.pt'), EXPERIMENTS['experiment_b_combined'])
# display(test_report['outcome'].value_counts())

## Processing contract

When raw data are available locally, the next cells will:

1. Create immutable per-sequence/video manifests and issue logs.
2. Validate Dataset 1 `exist`/`gt_rect` alignment and Dataset 2 decoded `.mat`/video alignment.
3. Render a reviewed sample of overlays. Overlay images are QA artifacts only; no boxes are drawn into model inputs.
4. Export validated thermal frames with separate YOLO labels and provenance metadata.
5. Apply deterministic temporal subsampling, duplicate checks, and recording-disjoint splits.
6. Stop for approval before any YOLO training.

Excluded Dataset 2 airplane and helicopter recordings will not be treated as unlabeled background. Dataset 1 `track1_test` and `track2_test` will remain inference-only.

## Planned model, evaluation, and outputs

After the dataset passes validation and training is approved, this notebook will add a lightweight maintained YOLO configuration, exact run commands, and fixed-seed Experiment A/B evaluation. The held-out test set will stay fixed and Dataset 2-only.

Expected result artifacts:

- `results/thermal-ir/metrics/test_metrics.json`
- `results/thermal-ir/figures/confusion_matrix.png`
- `results/thermal-ir/test_predictions.csv` — one row per held-out ground-truth object and unmatched prediction, including image ID, source, class, confidence, IoU/match status, and error type.

No metrics, confusion matrix, or predictions will be fabricated before a trained model is evaluated on the isolated test split.